# 04 - FPMC (Factorized Personalized Markov Chains)

FPMC scores *"given this traveller, currently in city `l`, how likely is city `i` next?"* by adding two factorized terms:

$$\text{score}(u,l,i) = \underbrace{\langle V^{UI}_u, V^{IU}_i\rangle}_{\text{personalization}} + \underbrace{\langle V^{LI}_l, V^{IL}_i\rangle}_{\text{Markov transition}}$$

The first term is matrix factorization (who prefers what); the second is a factorized Markov chain (what follows what). It is trained with **BPR**: for an observed transition `l -> i` push its score above that of a random city `j`.

All logic is in `src/models/fpmc.py`; this notebook runs the two ablations and logs results through the shared `evaluate.py`.

In [ ]:
import os, sys
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))
ROOT = find_root(); sys.path.insert(0, ROOT)
from src.preprocessing import load_processed, preprocess
try:
    data = load_processed(os.path.join(ROOT, 'data', 'processed'))
except FileNotFoundError:
    data = preprocess(data_dir=os.path.join(ROOT, 'data'), sample_n_trips=5000)
print('loaded:', data['meta'])

## Ablation - personalization axis: raw `user_id` vs `persona`

Classic FPMC personalizes by user. But in this dataset almost every `user_id` occurs in a single trip, so a per-user factor cannot generalise to unseen test users. We therefore also try a **persona** = (booker_country, device_class), a coarse profile that recurs across many trips. Comparing the two is a headline finding: it shows *where personalization signal actually lives* in this data.

In [ ]:
from src.models.fpmc import run_fpmc
from src.evaluate import append_result
RESULTS = os.path.join(ROOT, 'results.csv')

for axis in ['persona', 'user']:
    print(f'--- FPMC (axis={axis}) ---')
    metrics, _, val = run_fpmc(data, axis=axis, dim=64, epochs=20, verbose=True)
    print('TEST:', {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})
    append_result(RESULTS, f'FPMC-{axis}', metrics)

## The position feature

FPMC has no built-in notion of *where* in a trip a step sits (SASRec learns this for free via positional embeddings). We add a position-bucket embedding to the user side of the score, so the model can express e.g. "early in a trip vs the last leg home". Its effect is included in the numbers above; toggling it is a small extra ablation if time permits.

## What to write up

- FPMC should comfortably beat Popularity and Markov wherever the last-city transition and personalization carry signal.
- Report the **persona vs user** gap and explain it with the per-user sparsity measured in `01_data_exploration.ipynb`.
- On the full dataset, expect the raw-`user_id` variant to degrade (cold-start at test), which is the point of the ablation.